<a href="https://colab.research.google.com/github/RohanYashraj/ifoa-workshop/blob/main/notebooks_v2/01_genai_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · GenAI Basics — your first calls to the reasoner

**Agentic AI for Health Actuaries** · IAI Seminar · 25 August 2026 · Hub: `github.com/rohanyashraj/ifoa-workshop`

> All data in this notebook is **hypothetical** — ABC Health is a fictional entity calibrated to plausible Indian health insurance experience, for teaching only.

**Used in:** Session 1, Part 2 (The Reasoner).
**You will:** make your first Gemini API call, practise the CCCE prompt discipline, watch a hallucination happen on demand, and get guaranteed-parseable JSON out of an LLM.

**Setup (2 minutes):**
1. Get a free Gemini API key at [aistudio.google.com](https://aistudio.google.com) → *Get API key*.
2. In Colab, click the **key icon** (left sidebar) → *Add new secret* → name it `GOOGLE_API_KEY`, paste the key, toggle notebook access ON.
3. Run the cells top to bottom (`Runtime → Run all` after setup).

In [1]:
%pip install -q -U google-genai google-auth==2.49.0

Note: you may need to restart the kernel to use updated packages.


d:\Kasyap\Agentic AI Session\IAI\Notebooks\.venv\Scripts\python.exe: No module named pip


## §1 · Auth — the key never appears in the notebook
Colab Secrets keeps the key out of the notebook file. This is the same hygiene you will use for every agent you ship: secrets live in a store, never in code.

In [3]:
import os
from google import genai
from IPython.display import Markdown, display
from dotenv import load_dotenv
#from google.colab import userdata   # Colab-only; see comment below for local Jupyter

#os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# Local Jupyter alternative:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

client = genai.Client()
MODEL = "gemini-3.1-flash-lite"   # PINNED — silent model drift is an audit failure
print("Client ready, model pinned to:", MODEL)

Client ready, model pinned to: gemini-3.1-flash-lite


## §2 · First call — define IBNR for a board member

In [4]:
response = client.models.generate_content(
    model=MODEL,
    contents="Define IBNR for a non-actuarial board member, in one line.",
)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(response.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
print("\n--- usage ---")
print(response.usage_metadata)   # token counts: you will care about these when agents multiply call volume

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


📋 GEMINI MODEL RESPONSE


IBNR (Incurred But Not Reported) represents the estimated liability for claims that have already occurred but have not yet been formally reported to the company.


END OF MODEL RESPONSE

--- usage ---
cache_tokens_details=None cached_content_token_count=None candidates_token_count=31 candidates_tokens_details=None prompt_token_count=19 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=19
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=50 traffic_type=None


## §3 · CCCE — Clarity, Context, Constraints, Examples
The prompt below is the worked example from the slides: an IBNR commentary for ABC Health Q3 2024. Each bracketed fragment does exactly one job — edit any part without breaking the others.

**Exercise:** delete the Constraints block, re-run, and compare. Then rewrite the prompt for *your* line of business.

In [5]:
ccce_prompt = """
[Clarity] Write a two-paragraph commentary on the IBNR result for ABC Health Q3 2024.
[Context] Indemnity health book. Chain-ladder ultimate INR 186 Cr vs prior estimate INR 172 Cr.
Q3 saw a hospital network strike in two states.
[Constraints] Audience: appointed actuary peer-review meeting. Max 180 words.
Do not invent figures. Cite only the figures provided above.
[Example] Voice to match: "The Q2 ultimate of INR 164 Cr increased to INR 172 Cr after the network
expansion in Tier 2 cities..."
"""
resp = client.models.generate_content(model=MODEL, contents=ccce_prompt)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(resp.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE


The Q3 2024 IBNR valuation for the indemnity health book reflects an ultimate estimate of INR 186 Cr, representing an increase from the prior estimate of INR 172 Cr. This movement is primarily driven by the volatility introduced during the quarter, as the reported loss development patterns have deviated from historical expectations.

This upward adjustment captures the impact of a widespread hospital network strike occurring across two states. We observed a temporary suppression in claim reporting during the industrial action, followed by a surge in submission volume as operations normalized. Consequently, the chain-ladder methodology now incorporates higher loss development factors to account for the delayed reporting and the associated backlog, ensuring our reserves remain robust despite these atypical operational disruptions.


END OF MODEL RESPONSE


### §3.1 · Demo 1 — the vague version, for contrast
Run the deliberately vague prompt below, then re-run the CCCE version above and **diff the outputs**. Same model, same cost — the entire quality delta is the prompt.

In [6]:
vague = "Write about IBNR for our board."
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(client.models.generate_content(model=MODEL, contents=vague).text[:800]))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
# Expect: a generic essay that INVENTS plausible numbers (we gave it none)
# and lands in a register somewhere between textbook and LinkedIn.

📋 GEMINI MODEL RESPONSE


This briefing note is designed for a Board of Directors level, focusing on the strategic, financial, and risk-management implications of IBNR rather than the actuarial mechanics.

***

# Briefing Note: Understanding IBNR (Incurred But Not Reported)

### 1. Executive Summary
**IBNR** stands for **"Incurred But Not Reported."** In the context of our financial reporting, it represents the estimated liability for insurance claims that have already occurred but have not yet been formally reported to the company. 

Because we are legally and financially obligated to pay these claims, IBNR must be recorded as a liability on our balance sheet. It is a critical component of our capital adequacy and a primary driver of our financial volatility.

### 2. Why does IBNR exist?
There is almost always a "


END OF MODEL RESPONSE


### §3.2 · Demo 2 — one fact, two audiences
Audience is a prompt parameter. Same reserve-strengthening fact, rendered for a board member and for a new student. Note: the model *dresses* the fact we supply — it does not source it.

In [7]:
fact = ("We strengthened PMI hospitalisation reserves by INR 42 Cr "
        "following a sharp rise in empanelled-hospital tariffs.")

for audience, style in [
    ("board member", "2 sentences, business impact first, no jargon"),
    ("new actuarial student", "4 sentences, explain WHY tariffs drive PMI reserves, define terms"),
]:
    r = client.models.generate_content(
        model=MODEL,
        contents=f"Explain: {fact} For a {audience}. {style}")
    # Clear visual separation
    print("=" * 70)
    print("📋 GEMINI MODEL RESPONSE")
    print("=" * 70)

    display(Markdown(f"**{audience.upper()}**\n\n{r.text}"))

    print("\n" + "=" * 70)
    print("END OF MODEL RESPONSE")
    print("=" * 70)


📋 GEMINI MODEL RESPONSE


**BOARD MEMBER**

We have increased our hospitalisation reserves by INR 42 Cr to ensure we remain fully covered against recent significant hikes in healthcare provider costs. This proactive adjustment protects our balance sheet and ensures we have sufficient capital to meet all anticipated claims payouts.


END OF MODEL RESPONSE
📋 GEMINI MODEL RESPONSE


**NEW ACTUARIAL STUDENT**

Private Medical Insurance (PMI) reserves act as a financial buffer, representing the estimated future cost of valid claims that the insurer must set aside today. When empanelled hospitals—the network providers contracted to treat your policyholders—sharply increase their tariffs (the price list for procedures and room rents), the insurer’s projected cost per claim rises immediately. Because the insurer is legally required to hold sufficient capital to cover all anticipated liabilities, these higher price expectations necessitate an increase in the reserve balance. Consequently, the INR 42 Cr injection reflects an actuarial adjustment to ensure the company remains solvent against this higher medical inflation environment.


END OF MODEL RESPONSE


### §3.3 · Demo 3 — few-shot examples tame formatting
Show, don't tell: two worked examples buy you the delimiter, the casing, the arrow convention, and no chatty preamble. **Exercise:** feed it a genuinely weird input and see whether the pattern holds.

In [8]:
prompt = """Convert each change to the format of the examples.

EXAMPLES
In: Hospitalisation frequency moved from 3.2% to 3.5% for ages 45+.
Out: HOSP_FREQ | age 45+ | 3.2% -> 3.5%
In: Claim severity trend up 40bps.
Out: SEV_TREND | all | +40bps

NOW CONVERT
In: CI incidence for cardiac conditions, ages 40-55, moves from 0.45% to 0.52%.
Out:"""
print(client.models.generate_content(model=MODEL, contents=prompt).text)


Out: CI_INC_CARDIAC | age 40-55 | 0.45% -> 0.52%


### §3.4 · Demo 4 — step-by-step reasoning (with a warning label)
Asking for steps improves reliability — it does **not** guarantee it. Re-run this cell three times: do the running totals stay identical? This is why the afternoon's agent does arithmetic in *Python* and lets Gemini narrate.

In [9]:
prompt = """A PMI policy has base premium INR 9,000 with relativities:
age band 46-55 = 1.45, sum insured 10L = 1.30, family floater = 1.10, NCB 30% = 0.70.
Walk through the premium calculation STEP BY STEP, showing the running
total after each factor, then state the final premium."""
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(client.models.generate_content(model=MODEL, contents=prompt).text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
# Check by hand: 9000 * 1.45 * 1.30 * 1.10 * 0.70 = 13,063.05


📋 GEMINI MODEL RESPONSE


To calculate the final premium for the Private Medical Insurance (PMI) policy, we apply the relativities sequentially to the base premium. 

**Base Premium: INR 9,000**

### Step-by-Step Calculation:

**Step 1: Apply Age Band Relativity (46-55 = 1.45)**
*   Calculation: 9,000 × 1.45 = 13,050
*   **Running Total: INR 13,050**

**Step 2: Apply Sum Insured Relativity (10L = 1.30)**
*   Calculation: 13,050 × 1.30 = 16,965
*   **Running Total: INR 16,965**

**Step 3: Apply Family Floater Relativity (1.10)**
*   Calculation: 16,965 × 1.10 = 18,661.50
*   **Running Total: INR 18,661.50**

**Step 4: Apply NCB Relativity (30% discount = 0.70)**
*   Calculation: 18,661.50 × 0.70 = 13,063.05
*   **Running Total: INR 13,063.05**

***

### Final Premium:
The final calculated premium for the policy is **INR 13,063.05**.


END OF MODEL RESPONSE


### §3.5 · Demo review — the habit that IS the skill
1. CCCE moved quality more than a model upgrade would — specification beats horsepower.
2. Register control is leverage, but the model dresses facts; it doesn't source them.
3. Few-shot is a formatting contract — stress-test it before relying on it.
4. Step-by-step is transparency, not verified arithmetic.

**The loop:** prompt → output → review → edit prompt — the same loop you'll run on agent traces this afternoon.

## §4 · The hallucination demo — run it, believe it
We ask for a regulation that **does not exist**. The model will not say 'no such factor' — it will produce the most *plausible-sounding* answer, confidently.

⚠️ This exact failure mode reappears **inside your agent** in notebook 04 — and you will fix it with a guardrail tool.

In [10]:
hallucination_prompt = (
    "What is the IRDAI-mandated co-payment factor for senior-citizen PMI "
    "policies with sum insured above INR 10 lakh? "
    "Give the exact factor value and the section reference."
)
resp = client.models.generate_content(model=MODEL, contents=hallucination_prompt)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(resp.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
print("\n⚠️  Verify before you trust: there is no such published factor. "
      "Whatever appears above was constructed to be plausible, not true.")


📋 GEMINI MODEL RESPONSE


As of the latest regulatory updates from the Insurance Regulatory and Development Authority of India (IRDAI), there is **no mandated co-payment factor** for senior citizen policies with a sum insured above INR 10 lakh.

### Clarification on the Regulation:
Under the **IRDAI (Health Insurance) Regulations, 2024** (which superseded the erstwhile Health Insurance Regulations, 2016), the regulator has moved toward a more consumer-friendly framework. 

*   **Removal of Mandatory Co-pay:** IRDAI does not mandate a specific co-payment percentage for senior citizen policies. Co-payment is a product-level design feature determined by insurers based on their actuarial pricing, risk assessment, and underwriting guidelines.
*   **Prohibition on Age-Based Discrimination:** The new Master Circular on Health Insurance Products (notified in 2024) mandates that insurers **cannot deny the issuance of a health insurance policy to senior citizens** (aged 65 and above) and must provide options for products that cater to this segment. While insurers may still offer policies with co-payments to keep premiums affordable, the regulator has not set a "mandated factor" for high sum-insured policies.

### Important Context:
1.  **Market Practice:** While IRDAI does not mandate a co-pay, insurers often offer a "Co-pay Waiver" as an optional rider or build it into premium-tier plans. For policies with high sum insured (above INR 10 lakh), many insurers voluntarily reduce or eliminate co-pay requirements to compete for the senior citizen segment.
2.  **Section Reference:** You may refer to the **"Master Circular on Health Insurance Products"** (issued by IRDAI on May 29, 2024). Specifically, look at **Section 4 (Product Design)**, which governs the flexibility of insurers in designing products. The circular explicitly emphasizes the removal of barriers for senior citizens but does not prescribe a mandatory co-payment scale.
3.  **Previous Misconception:** In older industry discussions or specific state-sponsored schemes (like certain government-subsidized health covers), co-pays were common; however, in the **commercial private insurance market** governed by IRDAI, co-payment is a **contractual term** defined in the policy wording approved by the Authority during the "File and Use" process, not a fixed regulatory mandate.

**Recommendation:** If you are looking at a specific policy, the co-payment factor will be explicitly stated in the **Policy Schedule** or the **Product Brochure** under the "Terms and Conditions" section. If an insurer claims a co-pay is "mandated by IRDAI" for your specific sum insured, that statement is factually incorrect under current regulations.


END OF MODEL RESPONSE

⚠️  Verify before you trust: there is no such published factor. Whatever appears above was constructed to be plausible, not true.


## §5 · Structured output — because agents speak JSON
One config line guarantees parseable JSON. This is how every component of an agentic system exchanges data — prose is only for humans at the last step.

In [11]:
import json

prompt = """For individual PMI (health) cover, list 5 rating factors.
For each: name, direction (increase/decrease premium), one-line justification. Return JSON."""

resp = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"response_mime_type": "application/json"},
)
factors = json.loads(resp.text)   # guaranteed to parse
for f in factors:
    print(f)


{'name': 'Age', 'direction': 'increase', 'justification': 'Older individuals have a statistically higher probability of developing chronic health conditions requiring medical intervention.'}
{'name': 'Geographic Location', 'direction': 'increase', 'justification': 'Healthcare costs and hospital fees vary significantly by region, often based on the local cost of living and medical infrastructure.'}
{'name': 'Pre-existing Conditions', 'direction': 'increase', 'justification': 'Known medical history indicates a higher likelihood of future claims, necessitating a higher premium or specific policy exclusions.'}
{'name': 'Excess (Deductible) Amount', 'direction': 'decrease', 'justification': "A higher excess shifts a larger portion of initial claim costs to the policyholder, reducing the insurer's total liability."}
{'name': 'Tobacco Usage', 'direction': 'increase', 'justification': 'Smoking is linked to a wider range of serious health complications, resulting in higher long-term treatment c

## §6 · Review exercise — mark the model's homework
Treat the JSON above as a junior analyst's first draft and grade it:

1. Is every **direction** consistent with your priors?
2. Did it name factors your book doesn't collect (e.g. a wellness-programme discount, occupation class)?
3. What material factors are **missing** (BMI band? room-rent category?)
4. What would you still need before any of this goes near a tariff filing? *(Hint: magnitudes → a GLM run → notebook 02.)*

**The rule that survives today:** the reasoner narrates; tools know; humans sign.

---
**Log what you ran.** For anything regulatory: save the full prompt–response pair, the model id, and the timestamp — 'the AI wrote it' is not a defence without the receipt.

In [15]:
# Minimal call log — one CSV row per call. In production this is your observability stack.
import datetime, csv, pathlib

def log_call(prompt, response_text, model=MODEL, path="genai_call_log.csv"):
    new = not pathlib.Path(path).exists()
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["ts_utc", "model", "prompt", "response"])
        w.writerow([datetime.datetime.now(datetime.UTC).isoformat(), model, prompt, response_text])

log_call(prompt, resp.text)
print("logged — this habit is checklist question 10 in miniature")

logged — this habit is checklist question 10 in miniature
